# K-Means Clustering — Lloyd's Algorithm

**K-means** is one of the most widely used clustering algorithms. Given a point cloud $\{x_i\}_{i=1}^n \subset \mathbb{R}^d$ and a number of clusters $k$, it seeks to partition the data into $k$ groups and find a representative **centroid** $c_j$ for each group so as to minimise the **within-cluster sum of squares (WCSS)**:

$$
J(c, z) = \sum_{i=1}^n \|x_i - c_{z_i}\|^2, \quad z_i \in \{1,\ldots,k\}.
$$

The optimal partition induces a **Voronoi tessellation**: each cluster is the Voronoi cell of its centroid, and the centroids form a **centroidal Voronoi diagram** at convergence.

## Lloyd's iteration

**Lloyd's algorithm** (1982) alternates two steps:

1. **Assignment step**: assign each point to its nearest centroid,
   $$z_i = \arg\min_{j=1,\ldots,k} \|x_i - c_j\|^2.$$
2. **Update step**: recompute each centroid as the mean of its cluster,
   $$c_j = \frac{1}{|C_j|} \sum_{i:\, z_i=j} x_i.$$

Each step decreases (or preserves) the objective $J$:
- Assignment step: holds $c$ fixed, finds the global optimum in $z$.
- Update step: holds $z$ fixed; the mean minimises squared distance within each cluster.

Since $J \geq 0$ and is non-increasing, the algorithm converges in a finite number of steps to a **local minimum**.

## Voronoi tessellation

At any point in the algorithm, the assignment step partitions $\mathbb{R}^d$ into the Voronoi diagram of the current centroids:

$$
\mathcal{V}_j = \{x \in \mathbb{R}^d : \|x - c_j\| \leq \|x - c_l\|,\; \forall l \neq j\}.
$$

At convergence, every centroid is the mean of the data in its Voronoi cell — the **centroidal Voronoi property**.

This notebook:
- visualises the step-by-step Lloyd iteration with animated Voronoi cells;
- shows the monotone decrease of the objective;
- studies the effect of $k$ via the **elbow method**;
- demonstrates sensitivity to initialisation.

## Environment

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from ipywidgets import interact, IntSlider, Play, jslink
import ipywidgets as widgets

plt.rcParams['figure.dpi'] = 110
rng = np.random.default_rng(0)

## Dataset

We generate a mixture of 5 Gaussians with varying covariance scales to create a realistic non-trivial clustering problem.

In [2]:
def make_data(n=400, k=5, seed=42):
    rng_ = np.random.default_rng(seed)
    centres = rng_.uniform(-4, 4, (k, 2))
    sigmas  = rng_.uniform(0.4, 0.9, k)
    labels  = rng_.integers(0, k, n)
    X = centres[labels] + rng_.normal(0, sigmas[labels, None], (n, 2))
    return X, centres

X, true_c = make_data(400, 5, seed=42)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X[:, 0], X[:, 1], s=12, alpha=0.6, color='steelblue')
ax.scatter(true_c[:, 0], true_c[:, 1], c='r', s=100, marker='*', zorder=5, label='true centres')
ax.set_title('Input data (mixture of 5 Gaussians)')
ax.legend()
plt.tight_layout()
plt.savefig('kmeans_data.png', bbox_inches='tight')
plt.close()
print('Saved kmeans_data.png')

Saved kmeans_data.png


## Core Lloyd implementation

We implement Lloyd's algorithm while recording the full history of centroids and assignments at each iteration, enabling step-by-step visualisation.

In [3]:
def lloyd_history(X, centres_init, max_iter=30):
    """Run Lloyd and record history of (centres, assignments, cost)."""
    centres = centres_init.copy()
    history = []
    for _ in range(max_iter):
        dists = np.sum((X[:, None] - centres[None])**2, axis=2)
        z     = np.argmin(dists, axis=1)
        cost  = dists[np.arange(len(X)), z].sum()
        history.append((centres.copy(), z.copy(), cost))

        new_c = np.array([
            X[z == j].mean(axis=0) if (z == j).any() else centres[j]
            for j in range(len(centres))
        ])
        if np.allclose(centres, new_c):
            break
        centres = new_c
    # Final step
    dists = np.sum((X[:, None] - centres[None])**2, axis=2)
    z = np.argmin(dists, axis=1)
    cost = dists[np.arange(len(X)), z].sum()
    history.append((centres.copy(), z.copy(), cost))
    return history

k_main = 5
rng_init = np.random.default_rng(7)
c0 = X[rng_init.choice(len(X), k_main, replace=False)]
history = lloyd_history(X, c0)
print(f'Converged in {len(history)} iterations')

Converged in 12 iterations


## Step-by-step Voronoi visualisation

Each panel shows a snapshot of the Voronoi cells (background), coloured data points (current assignment), and centroids (stars) at a given Lloyd iteration. Watching the centroids migrate and the cells reshape makes the algorithm intuitive.

In [4]:
def draw_voronoi_snap(ax, X, centres, z, cost, title=''):
    k = len(centres)
    xl = X[:, 0].min()-1; xr = X[:, 0].max()+1
    yl = X[:, 1].min()-1; yr = X[:, 1].max()+1
    xg = np.linspace(xl, xr, 200)
    yg = np.linspace(yl, yr, 200)
    Xg, Yg = np.meshgrid(xg, yg)
    grid = np.c_[Xg.ravel(), Yg.ravel()]
    dg   = np.sum((grid[:, None] - centres[None])**2, axis=2)
    Zg   = np.argmin(dg, axis=1).reshape(Xg.shape)
    cmap = plt.cm.Set2
    ax.contourf(Xg, Yg, Zg, levels=np.arange(-0.5, k+0.5),
                cmap=cmap, alpha=0.35)
    for j in range(k):
        mask = z == j
        ax.scatter(X[mask, 0], X[mask, 1], s=10, alpha=0.8,
                   color=cmap(j / k), edgecolors='none')
    ax.scatter(centres[:, 0], centres[:, 1],
               c='k', s=120, marker='*', zorder=5)
    ax.set_xlim(xl, xr); ax.set_ylim(yl, yr)
    ax.set_title(title, fontsize=9)

snap_iters = [0, 1, 2, 4, len(history)-1]
fig, axes = plt.subplots(1, len(snap_iters), figsize=(16, 3.8))
for ax, it in zip(axes, snap_iters):
    c, z, cost = history[it]
    draw_voronoi_snap(ax, X, c, z, cost,
                      title=f'iter {it}  WCSS={cost:.1f}')
fig.suptitle('Lloyd\'s algorithm — Voronoi snapshots', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('lloyd_snapshots.png', bbox_inches='tight')
plt.close()
print('Saved lloyd_snapshots.png')

Saved lloyd_snapshots.png


## Objective convergence

The WCSS objective is guaranteed to be **non-increasing** at every step. In practice it typically decreases rapidly in the first few iterations and then plateaus. The curve is piecewise-decreasing with flat segments corresponding to assignment steps that did not change any label.

In [5]:
costs = [h[2] for h in history]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(costs, 'o-', color='steelblue', lw=2, ms=6)
ax.set_xlabel('iteration'); ax.set_ylabel('WCSS')
ax.set_title('Lloyd\'s algorithm — objective convergence')
ax.set_yscale('log')
plt.tight_layout()
plt.savefig('wcss_convergence.png', bbox_inches='tight')
plt.close()
print('Saved wcss_convergence.png')

Saved wcss_convergence.png


## Elbow method: choosing $k$

K-means requires specifying $k$ in advance. The **elbow method** plots the minimum WCSS (over multiple restarts) as a function of $k$. The objective always decreases as $k$ grows; the optimal $k$ is identified at the **elbow** — the point of diminishing returns where adding one more cluster gives a small relative gain.

Formally, we look for the $k$ that maximises the second difference $\Delta^2 J(k)$, or equivalently where the relative decrease $\frac{J(k)-J(k+1)}{J(k)}$ first becomes small.

In [6]:
def best_lloyd(X, k, n_restarts=8, seed=0):
    best = np.inf
    for s in range(n_restarts):
        rng_ = np.random.default_rng(s + seed)
        c0_ = X[rng_.choice(len(X), k, replace=False)]
        hist_ = lloyd_history(X, c0_, max_iter=50)
        cost_ = hist_[-1][2]
        if cost_ < best:
            best = cost_
    return best

k_range = range(1, 11)
wcss_k  = [best_lloyd(X, k) for k in k_range]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(list(k_range), wcss_k, 'o-', color='steelblue', lw=2, ms=7)
ax.axvline(5, color='crimson', ls='--', lw=1.5, label='true $k=5$')
ax.set_xlabel('$k$'); ax.set_ylabel('WCSS (best of 8 restarts)')
ax.set_title('Elbow method — choosing $k$')
ax.legend()
plt.tight_layout()
plt.savefig('elbow.png', bbox_inches='tight')
plt.close()
print('Saved elbow.png')

Saved elbow.png


## Effect of initialisation

Because the objective is non-convex, different random initialisations can lead to very different local minima. We run 6 restarts from the same data, display each converged Voronoi diagram, and annotate the final WCSS to show the variance in solution quality.

In [7]:
n_show = 6
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for idx, ax in enumerate(axes.ravel()):
    rng_ = np.random.default_rng(idx * 17 + 3)
    c0_ = X[rng_.choice(len(X), k_main, replace=False)]
    hist_ = lloyd_history(X, c0_, max_iter=50)
    c_, z_, cost_ = hist_[-1]
    draw_voronoi_snap(ax, X, c_, z_, cost_,
                      title=f'restart {idx+1}  WCSS={cost_:.1f}')
fig.suptitle('K-means sensitivity to initialisation ($k=5$)', fontsize=12)
plt.tight_layout()
plt.savefig('init_sensitivity.png', bbox_inches='tight')
plt.close()
print('Saved init_sensitivity.png')

Saved init_sensitivity.png


## Interactive Lloyd step viewer

The slider below steps through the Lloyd iterations, showing the Voronoi partition evolving from the initial seeds to the converged solution.

## Static snapshot

In [8]:
STATIC_SNAPSHOT = True
if STATIC_SNAPSHOT:
    snap_iters_small = [0, 1, 3, len(history)-1]
    fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
    for ax, it in zip(axes, snap_iters_small):
        c, z, cost = history[it]
        draw_voronoi_snap(ax, X, c, z, cost,
                          title=f'iter {it}  WCSS={cost:.1f}')
    plt.tight_layout()
    plt.savefig('snippet.png', bbox_inches='tight')
    plt.close()
    print('Saved snippet.png')

Saved snippet.png


## Takeaways

- Lloyd's algorithm is a coordinate-descent method on the joint space of assignments and centroids; it is guaranteed to converge in finitely many steps but only to a **local minimum**.
- The converged partition is a **centroidal Voronoi tessellation**: each cluster cell contains exactly the data points closest to its mean.
- The **elbow method** provides a heuristic for choosing $k$, though it requires care — the elbow may be ambiguous for real data.
- Running multiple restarts and selecting the best solution is essential to mitigate initialisation sensitivity.

## Bibliography

- Lloyd, S. P. (1982). Least squares quantization in PCM. *IEEE Trans. Inf. Theory*, 28(2), 129–137.
- MacQueen, J. (1967). Some methods for classification and analysis of multivariate observations. *Proc. 5th Berkeley Symp. Math. Stat. Prob.*, 281–297.
- Du, Q., Faber, V., and Gunzburger, M. (1999). Centroidal Voronoi tessellations: applications and algorithms. *SIAM Review*, 41(4), 637–676.
- Steinhaus, H. (1956). Sur la division des corps matériels en parties. *Bull. Acad. Polon. Sci.*, 4(12), 801–804.